In [ ]:
# Lab type: debug
# Course: AI-Assisted Data Science
# Lesson: Silent Failure Patterns in AI-Generated Pipelines
# Task: Detect and fix six silent failure patterns in runnable but incorrect AI-generated code

# Lab: Silent Failure Patterns

Each section presents a piece of AI-generated code that **runs without errors** but produces
incorrect results. Your task for each pattern is to:

1. Run the broken code and observe the output
2. Use the detection cells to identify what went wrong
3. Write the corrected version

Work through the patterns in order — later patterns build on skills from earlier ones.

## Setup

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/SophiArch/notebooks/main/datasets/employee_records.csv"
employees = pd.read_csv(url)

print(employees.shape)
employees.head()

---
## Pattern 1: Fitting transformers on the full dataset

The AI generated this preprocessing pipeline for a salary-prediction model.
Run the cell, then answer the questions below.

In [ ]:
# --- AI-GENERATED CODE (broken) ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df1 = employees[["years_exp", "salary"]].dropna().copy()

X1 = df1[["years_exp"]]
y1 = df1["salary"]

# Scale before splitting
scaler1 = StandardScaler()
X1_scaled = scaler1.fit_transform(X1)   # <-- fit on entire dataset

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1_scaled, y1, test_size=0.2, random_state=42
)

print(f"Scaler mean (full dataset): {scaler1.mean_[0]:.4f}")
print(f"Train rows: {len(X1_train)}, Test rows: {len(X1_test)}")

### Detect it

If the scaler was fit correctly (on training data only), its learned mean should match
the training set mean — **not** the full dataset mean.
Run the cell below to compare.

In [ ]:
# What the scaler actually learned
print(f"Scaler mean (as fitted):          {scaler1.mean_[0]:.4f}")

# What the training set mean actually is
# Reverse-map X1_train back to original years_exp values
train_mean_years_exp = X1["years_exp"].iloc[
    train_test_split(range(len(X1)), test_size=0.2, random_state=42)[0]
].mean()
print(f"Actual training set mean:         {train_mean_years_exp:.4f}")
print(f"Full dataset mean:                {X1['years_exp'].mean():.4f}")

print("\nThe scaler mean matches the full dataset mean, not the training mean.")
print("Test-set statistics contaminated the scaler.")

> **Question:** Why does a scaler fit on the full dataset make the model's test-set evaluation
> look better than it really is?

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: rewrite the pipeline so the scaler is fit on training data only
# Verify by printing the scaler mean alongside the training set mean

scaler1_fixed = StandardScaler()

# Split first
X1_train_raw, X1_test_raw, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

# Your fix here:
# X1_train_scaled = ...
# X1_test_scaled  = ...

---
## Pattern 2: Target leakage through derived features

We're building a model to predict whether an employee will leave (`left`)
based on data available at annual review time.
The AI added a `days_employed` feature from an `exit_date` column.
Run the cell, then examine the problem.

In [ ]:
# Synthetic dataset with a leaky column
rng = np.random.default_rng(7)
n = len(employees)

df2 = employees[["years_exp", "salary", "region"]].copy()

# Binary target: did the employee leave within 12 months of review?
df2["left"] = (rng.random(n) < 0.18).astype(int)

# exit_date is only populated for employees who left
review_date = pd.Timestamp("2024-01-01")
df2["exit_date"] = pd.NaT
leaver_mask = df2["left"] == 1
df2.loc[leaver_mask, "exit_date"] = review_date + pd.to_timedelta(
    rng.integers(10, 340, size=leaver_mask.sum()), unit="D"
)

print(df2[["left", "exit_date"]].head(10))
print(f"\nRows with exit_date populated: {df2['exit_date'].notna().sum()}")
print(f"Rows where left == 1:          {df2['left'].sum()}")

In [ ]:
# --- AI-GENERATED CODE (broken) ---

# Engineer a feature: days between review and exit
df2["days_to_exit"] = (df2["exit_date"] - review_date).dt.days.fillna(-1)

print(df2[["left", "exit_date", "days_to_exit"]].head(10))
print(f"\nCorrelation of days_to_exit with target 'left':")
print(df2[["left", "days_to_exit"]].corr().round(3))

### Detect it

A feature that perfectly predicts the target — or is only populated when the target is 1 —
is a strong signal of leakage.

In [ ]:
# days_to_exit is -1 for non-leavers and a positive number for leavers
# Check: how well does it predict the target on its own?
print("days_to_exit distribution by target:")
print(df2.groupby("left")["days_to_exit"].value_counts().head(20))

# Check: what proportion of rows with days_to_exit > 0 have left == 1?
leaked_rows = df2[df2["days_to_exit"] > 0]
print(f"\nRows where days_to_exit > 0: {len(leaked_rows)}")
print(f"Of those, left == 1:          {leaked_rows['left'].sum()} ({leaked_rows['left'].mean():.0%})")

> **Question:** `days_to_exit` is only available after the employee leaves.
> At inference time (annual review for active employees), what value would this feature have?
> What would a model trained on this feature actually learn?

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: remove the leaky feature and any other columns derived from post-event data
# Print the final column list to confirm exit_date and days_to_exit are excluded

# df2_clean = ...

---
## Pattern 3: Silent type coercion

The dataset has been received from an external system where `salary` was exported as strings
for some rows. The AI added a `fillna` + `astype(int)` step.

In [ ]:
# Simulate a CSV export where some salary values are strings with currency symbols
df3 = employees[["employee_id", "salary"]].copy().astype({"salary": object})

rng3 = np.random.default_rng(13)
messy_idx = rng3.choice(df3.index, size=80, replace=False)
# Some rows: salary as a string with currency formatting
df3.loc[messy_idx[:40], "salary"] = df3.loc[messy_idx[:40], "salary"].apply(
    lambda v: f"£{float(v):,.0f}"
)
# Some rows: genuinely missing
df3.loc[messy_idx[40:], "salary"] = np.nan

print("Sample of mixed salary column:")
print(df3["salary"].head(20).to_string())
print(f"\ndtype: {df3['salary'].dtype}")

In [ ]:
# --- AI-GENERATED CODE (broken) ---
df3_processed = df3.copy()

# Convert to numeric, fill missing, cast to int
df3_processed["salary_numeric"] = pd.to_numeric(
    df3_processed["salary"], errors="coerce"
)
df3_processed["salary_numeric"] = df3_processed["salary_numeric"].fillna(
    df3_processed["salary_numeric"].mean()
)
df3_processed["salary_clean"] = df3_processed["salary_numeric"].astype(int)

print(df3_processed[["salary", "salary_numeric", "salary_clean"]].head(10))

### Detect it

The code ran without errors. But `pd.to_numeric(..., errors='coerce')` silently converts
unparseable strings to `NaN` — meaning the currency-formatted rows are now missing,
not just the originally-missing rows.

In [ ]:
# How many rows were lost to coercion vs genuinely missing?
originally_missing = df3["salary"].isna().sum()
after_coerce_missing = df3_processed["salary_numeric"].isna().sum()
# (fillna was already applied, so check a version before fillna)

coerced = df3.copy()
coerced["salary_numeric"] = pd.to_numeric(coerced["salary"], errors="coerce")
coerce_induced_missing = coerced["salary_numeric"].isna().sum() - originally_missing

print(f"Originally missing:                 {originally_missing}")
print(f"Additional NaNs from coercion:      {coerce_induced_missing}")
print(f"(These rows had parseable strings — their values were silently lost)")

# Also: astype(int) truncates floats — show the rounding loss
sample_floats = df3_processed["salary_numeric"].dropna().head(5)
print(f"\nSample float → int truncation:")
for f in sample_floats:
    print(f"  {f:.2f}  →  {int(f)}")

> **Question:** The AI used `errors='coerce'` silently. How would you know from the output
> alone that 40 salary values were coerced to NaN and then filled with the mean?
> What check should you run before and after any type conversion?

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: parse the currency-formatted strings correctly before conversion
# Steps:
#   1. Strip the '£' and commas from string values before calling pd.to_numeric
#   2. Check before and after how many NaN values exist
#   3. Fill missing values with the median (more robust than mean)
#   4. Keep salary as float — only cast to int if there is a clear reason

df3_fixed = df3.copy()

# Your fix here:
# df3_fixed["salary_clean"] = ...

---
## Pattern 4: Groupby silently dropping NaN keys

The AI generated a regional salary summary. We've injected some missing `region` values
to simulate a real-world data quality issue.

In [ ]:
# Inject missing region values
df4 = employees[["region", "salary"]].copy()
rng4 = np.random.default_rng(99)
missing_region_idx = rng4.choice(df4.index, size=47, replace=False)
df4.loc[missing_region_idx, "region"] = np.nan

print("Region value counts (including NaN):")
print(df4["region"].value_counts(dropna=False))

In [ ]:
# --- AI-GENERATED CODE (broken) ---
regional_salary = df4.groupby("region")["salary"].agg(["mean", "count"]).round(0)
print(regional_salary)
print(f"\nTotal rows accounted for: {regional_salary['count'].sum()} of {len(df4)}")

### Detect it

Compare the total row count in the aggregation against the full dataset length.

In [ ]:
rows_in_result = regional_salary["count"].sum()
rows_in_data   = len(df4)
rows_dropped   = rows_in_data - rows_in_result

print(f"Rows in aggregation:  {rows_in_result}")
print(f"Rows in dataset:      {rows_in_data}")
print(f"Rows silently dropped: {rows_dropped}")
print(f"\nThose dropped rows have total salary: £{df4.loc[missing_region_idx, 'salary'].sum():,.0f}")

> **Question:** The dropped rows' salaries were excluded from every region's mean.
> In what scenarios would this distort your analysis most severely?
> (Think about whether missing `region` is likely to be random, or correlated with something.)

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: include NaN as an explicit group in the aggregation
# Verify: the row counts should now sum to len(df4)

# regional_salary_fixed = ...

---
## Pattern 5: Random splitting of time-series data

We have monthly employee headcount data. The AI generated a train/test split
for forecasting next month's headcount.

In [ ]:
# Synthetic monthly headcount time series (36 months)
rng5 = np.random.default_rng(55)
dates = pd.date_range(start="2021-01", periods=36, freq="MS")
trend = np.linspace(200, 280, 36)
noise = rng5.normal(0, 5, 36)

ts = pd.DataFrame({
    "month": dates,
    "headcount": (trend + noise).round().astype(int),
    "avg_salary": (50000 + np.linspace(0, 8000, 36) + rng5.normal(0, 800, 36)).round(-2)
})

print(ts.head())
print("...")
print(ts.tail())

In [ ]:
# --- AI-GENERATED CODE (broken) ---
from sklearn.model_selection import train_test_split

X5 = ts[["avg_salary"]]
y5 = ts["headcount"]

# Random split — does not preserve time order
X5_train, X5_test, y5_train, y5_test = train_test_split(
    X5, y5, test_size=0.2, random_state=42
)

print(f"Train indices (first 5): {sorted(X5_train.index.tolist())[:5]}")
print(f"Test  indices (first 5): {sorted(X5_test.index.tolist())[:5]}")

### Detect it

For a valid time-series split, every training date must be earlier than every test date.

In [ ]:
train_months = ts.loc[X5_train.index, "month"]
test_months  = ts.loc[X5_test.index,  "month"]

print(f"Train date range: {train_months.min().date()} → {train_months.max().date()}")
print(f"Test  date range: {test_months.min().date()}  → {test_months.max().date()}")

# Chronological ordering check
contaminated = train_months.max() >= test_months.min()
print(f"\nFuture data in training set: {contaminated}")

# Count how many test months appear before the train max
future_leak_count = (test_months < train_months.max()).sum()
print(f"Test rows chronologically before train max: {future_leak_count} of {len(test_months)}")

> **Question:** A forecasting model trained this way would see 2023 headcount data
> during training and be 'tested' on 2021 data. Why does this make the model's
> evaluation metrics unrealistically good — and fail silently in production?

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: implement a chronological split (80% train, 20% test)
# Then assert that the split is clean: train max date < test min date

ts_sorted = ts.sort_values("month").reset_index(drop=True)
split_idx = int(len(ts_sorted) * 0.8)

# Your fix here:
# train5 = ...
# test5  = ...

# assert train5["month"].max() < test5["month"].min(), "Split is contaminated"

---
## Pattern 6: `fillna(0)` applied to all columns

The AI used a single `fillna(0)` call to handle all missing values across the dataset.

In [ ]:
# Inject missing values into several columns of different types
df6 = employees.copy()
rng6 = np.random.default_rng(21)

idx_salary = rng6.choice(df6.index, size=35, replace=False)
idx_dept   = rng6.choice(df6.index, size=28, replace=False)
idx_perf   = rng6.choice(df6.index, size=20, replace=False)

df6.loc[idx_salary, "salary"]     = np.nan
df6.loc[idx_dept,   "department"] = np.nan
df6.loc[idx_perf,   "performance"] = np.nan

print("Missing values before fillna:")
print(df6.isna().sum()[df6.isna().sum() > 0])
print(f"\ndepartment dtype: {df6['department'].dtype}")

In [ ]:
# --- AI-GENERATED CODE (broken) ---
df6_filled = df6.fillna(0)

print("Missing values after fillna(0):")
print(df6_filled.isna().sum())
print("\nAll zeros — looks clean!")

### Detect it

`isna().sum()` shows zero missing values — but the categorical columns now contain
the string `"0"`, which is a new category that shouldn't exist.

In [ ]:
# Check the department column after fillna
print("department value_counts after fillna(0):")
print(df6_filled["department"].value_counts(dropna=False))

print("\nperformance value_counts after fillna(0):")
print(df6_filled["performance"].value_counts(dropna=False))

print("\nNotice '0' appears as a category — this is not a real department or performance level.")

In [ ]:
# Also check: what does fillna(0) do to a numeric column with valid zeros?
# years_exp legitimately contains 0 for new hires — fillna(0) is coincidentally OK there,
# but only by accident. Check the salary column.
print("Salary statistics after fillna(0):")
print(df6_filled["salary"].describe().round(0))
print(f"\nRows where salary == 0: {(df6_filled['salary'] == 0).sum()}")
print("(These are filled missing values, not real £0 salaries.)")

> **Question:** If this dataset were fed to a model after `fillna(0)`, what would the model
> learn from the `department == '0'` rows and `salary == 0` rows?
> How would this corrupt a salary-prediction model?

*(Write your answer here.)*

### Fix it

In [ ]:
# TODO: apply an appropriate fill strategy to each column type
#   - salary:      fill with the median
#   - department:  fill with the mode (most common department)
#   - performance: fill with 'meets_expectations' (domain default)
# Then verify with value_counts that no '0' string values appear in categorical columns

df6_fixed = df6.copy()

# Your fix here:
# df6_fixed["salary"]     = ...
# df6_fixed["department"] = ...
# df6_fixed["performance"] = ...

---
## Summary checks

Run this cell to confirm all your fixes produced clean outputs.

In [ ]:
checks = {}

# Pattern 1: scaler fitted on train only
try:
    checks["P1 – scaler fit on train only"] = abs(scaler1_fixed.mean_[0] - X1_train_raw["years_exp"].mean()) < 0.01
except Exception:
    checks["P1 – scaler fit on train only"] = "NOT YET FIXED"

# Pattern 2: leaky columns removed
try:
    checks["P2 – exit_date removed"] = "exit_date" not in df2_clean.columns and "days_to_exit" not in df2_clean.columns
except Exception:
    checks["P2 – exit_date removed"] = "NOT YET FIXED"

# Pattern 3: no salary values lost to coercion
try:
    checks["P3 – no coercion-induced NaN"] = df3_fixed["salary_clean"].isna().sum() == 0
except Exception:
    checks["P3 – no coercion-induced NaN"] = "NOT YET FIXED"

# Pattern 4: groupby row count matches dataset
try:
    checks["P4 – all rows in aggregation"] = regional_salary_fixed["count"].sum() == len(df4)
except Exception:
    checks["P4 – all rows in aggregation"] = "NOT YET FIXED"

# Pattern 5: chronological split
try:
    checks["P5 – chronological split"] = train5["month"].max() < test5["month"].min()
except Exception:
    checks["P5 – chronological split"] = "NOT YET FIXED"

# Pattern 6: no '0' string in categoricals
try:
    checks["P6 – no '0' in categoricals"] = (
        "0" not in df6_fixed["department"].values and
        "0" not in df6_fixed["performance"].values
    )
except Exception:
    checks["P6 – no '0' in categoricals"] = "NOT YET FIXED"

print("=" * 50)
for name, result in checks.items():
    status = "PASS" if result is True else ("FAIL" if result is False else result)
    print(f"{status:>14}  {name}")
print("=" * 50)

## Reflection

Each of the six patterns above ran without raising any exception.
Write 3–5 bullet points on the general principle each one illustrates
about AI-generated data code.

*(Write your reflection here.)*

- 
- 
- 